In [6]:
import os
import glob
import importlib
import pprint
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyspark
import pyspark.sql.functions as F
from pyspark.sql.functions import col

import utils.airflow_model_inference as model_inference
importlib.reload(model_inference)   # pick up edits to utils/airflow_model_inference.py without restarting the kernel

<module 'model_inference' from '/app/model_inference.py'>

In [2]:
# Build a .py script that takes a snapshot date, loads a model artefact and make an inference and save to datamart
# snapshot_date = application month in the GOLD feature store (datamart/gold/feature_store/)

## set up pyspark session

In [7]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 05:35:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [8]:
snapshot_date_str = "2024-01-01"
model_name = "credit_model_2024_09_01.pkl"


In [9]:
config = model_inference.build_config(snapshot_date_str, model_name)
pprint.pprint(config)

{'feature_store_directory': 'datamart/gold/feature_store/',
 'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 1, 1, 0, 0),
 'snapshot_date_str': '2024-01-01'}


## load model artefact from model bank

In [10]:
model_artefact = model_inference.load_model_artefact(config)

print("model_version:", model_artefact["model_version"], "| type:", model_artefact["model_type"],
      "| label:", model_artefact["label_def"])
print("trained on labels", model_artefact["data_dates"]["train_test_start_date"].date(), "->",
      model_artefact["data_dates"]["oot_end_date"].date())
print(len(model_artefact["feature_cols"]), "input features")
pprint.pprint(model_artefact["results"])

Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
model_version: credit_model_2024_09_01 | type: xgboost | label: 30dpd_6mob
trained on labels 2023-07-01 -> 2024-08-31
57 input features
{'auc_oot': 0.8334,
 'auc_test': 0.8458,
 'auc_train': 0.9787,
 'cv_auc': 0.813,
 'gini_oot': 0.667,
 'gini_test': 0.692,
 'gini_train': 0.957}


## load gold feature store
One partition per application month: `datamart/gold/feature_store/gold_feature_store_YYYY_MM_DD.parquet`
(attributes + financials + clickstream + engineered features, PII already dropped).

In [11]:
features_pdf = model_inference.load_features(spark, config)
features_pdf

extracted features_sdf 485 2024-01-01 00:00:00 from datamart/gold/feature_store/gold_feature_store_2024_01_01.parquet


,Customer_ID,snapshot_date,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,...,n_Student_Loan,Total_Loans,n_secured,n_unsecured,n_distinct_loan_types,debt_to_income,emi_to_income,Spend_Level,Payment_Value,Credit_Mix_MinAmt
0,CUS_0x102d,2024-01-01,31.0,Entrepreneur,89064.52,7256.04,5,3,1,1.0,...,0,1,1,0,1,0.007280,0.005178,High,Medium,__No
1,CUS_0x1051,2024-01-01,42.0,Engineer,35022.22,2859.52,3,5,4,1.0,...,1,1,0,1,1,0.028566,0.007417,Low,Small,Good_No
2,CUS_0x1269,2024-01-01,22.0,Manager,42031.09,3762.59,2,1,8,3.0,...,0,3,1,0,3,0.001988,0.017677,High,Medium,Good_No
3,CUS_0x1290,2024-01-01,31.0,Architect,10455.88,729.32,6,5,12,3.0,...,0,3,0,3,3,0.007953,0.035307,Low,Medium,Good_No
4,CUS_0x12d1,2024-01-01,41.0,Accountant,21384.94,1646.08,4,4,15,2.0,...,0,2,0,1,2,0.032337,0.013426,Low,Medium,Standard_Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,CUS_0xc68d,2024-01-01,41.0,Musician,15497.02,1193.42,8,4,19,2.0,...,0,2,1,0,2,0.076180,0.020253,Low,Medium,Standard_Yes
481,CUS_0xc6d8,2024-01-01,34.0,Developer,120955.92,10021.66,2,7,4,1.0,...,0,1,0,0,1,0.003054,0.005739,High,Large,Good_No
482,CUS_0xe8d,2024-01-01,26.0,Journalist,14441.51,1453.46,9,8,26,2.0,...,1,2,0,2,2,0.092533,0.014469,High,Medium,Bad_Yes
483,CUS_0xf9e,2024-01-01,43.0,Teacher,142475.32,12013.94,8,5,11,2.0,...,0,2,2,0,2,0.004412,0.010355,High,Large,__No


## preprocess data for modeling
The artefact's `model` is a full sklearn `Pipeline` (imputation, one-hot encoding, scaling where needed + model),
fitted on the training set. It takes the raw gold columns in `model_artefact["feature_cols"]` - no manual scaler step.

In [12]:
# prepare X_inference - raw gold feature columns, in the order the model was trained on
X_inference = features_pdf[model_artefact["feature_cols"]]
print('X_inference', X_inference.shape)
print('clickstream missing:', round(X_inference.filter(like="fe_").isna().all(axis=1).mean(), 3))

# what the model actually sees after the pipeline's preprocessing step
preprocess = model_artefact["model"].named_steps["preprocess"]
pd.DataFrame(preprocess.transform(X_inference), columns=preprocess.get_feature_names_out()).head()

X_inference (485, 57)
clickstream missing: 0.0


,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,...,Credit_Mix_Unknown,Payment_of_Min_Amount_No,Payment_of_Min_Amount_Yes,Spend_Level_High,Spend_Level_Low,Spend_Level_Unknown,Payment_Value_Large,Payment_Value_Medium,Payment_Value_Small,Payment_Value_Unknown
0,31.0,89064.52,7256.04,5.0,3.0,1.0,1.0,6.0,5.0,6.37,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,42.0,35022.22,2859.52,3.0,5.0,4.0,1.0,6.0,11.0,0.00,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,22.0,42031.09,3762.59,2.0,1.0,8.0,3.0,1.0,5.0,14.11,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,31.0,10455.88,729.32,6.0,5.0,12.0,3.0,11.0,19.0,9.71,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,41.0,21384.94,1646.08,4.0,4.0,15.0,2.0,24.0,15.0,14.37,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


## model prediction inference

In [13]:
y_inference_pdf = model_inference.predict(model_artefact, features_pdf, config)
y_inference_pdf

X_inference (485, 57) | clickstream missing: 0.0


,Customer_ID,snapshot_date,model_name,model_predictions
0,CUS_0x102d,2024-01-01,credit_model_2024_09_01.pkl,0.232482
1,CUS_0x1051,2024-01-01,credit_model_2024_09_01.pkl,0.214278
2,CUS_0x1269,2024-01-01,credit_model_2024_09_01.pkl,0.030795
3,CUS_0x1290,2024-01-01,credit_model_2024_09_01.pkl,0.217427
4,CUS_0x12d1,2024-01-01,credit_model_2024_09_01.pkl,0.254111
...,...,...,...,...
480,CUS_0xc68d,2024-01-01,credit_model_2024_09_01.pkl,0.217137
481,CUS_0xc6d8,2024-01-01,credit_model_2024_09_01.pkl,0.596566
482,CUS_0xe8d,2024-01-01,credit_model_2024_09_01.pkl,0.552772
483,CUS_0xf9e,2024-01-01,credit_model_2024_09_01.pkl,0.646383


## save model inference to datamart gold table

In [14]:
filepath = model_inference.save_predictions(spark, y_inference_pdf, config)

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_01_01.parquet


## backfill

In [15]:
# set up config - gold feature store covers application months 2023-01-01 .. 2025-01-01
start_date_str = "2023-01-01"
end_date_str = "2024-06-01"

In [16]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)


In [17]:
# pass the notebook's spark session so the job doesn't stop it
for snapshot_date in dates_str_lst:
    print(snapshot_date)
    model_inference.main(snapshot_date, model_name, spark=spark)

2023-01-01


---starting job---


{'feature_store_directory': 'datamart/gold/feature_store/',
 'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 1, 1, 0, 0),
 'snapshot_date_str': '2023-01-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 530 2023-01-01 00:00:00 from datamart/gold/feature_store/gold_feature_store_2023_01_01.parquet
X_inference (530, 57) | clickstream missing: 0.0
saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_01_01.parquet


---completed job---


2023-02-01


---starting job---


{'feature_store_directory': 'datamart/gold/feature_store/',
 'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.date

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_11_01.parquet


---completed job---


2023-12-01


---starting job---


{'feature_store_directory': 'datamart/gold/feature_store/',
 'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 12, 1, 0, 0),
 'snapshot_date_str': '2023-12-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 489 2023-12-01 00:00:00 from datamart/gold/feature_store/gold_feature_store_2023_12_01.parquet
X_inference (489, 57) | clickstream missing: 0.0
saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_12_01.parquet


---completed job---


2024-01-01


---starting job---


{'feature_store_directory': 'datamart/gold/feature_store/',
 'model_artefact_filepath': 'model_bank/c

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_04_01.parquet


---completed job---


2024-05-01


---starting job---


{'feature_store_directory': 'datamart/gold/feature_store/',
 'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 5, 1, 0, 0),
 'snapshot_date_str': '2024-05-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 491 2024-05-01 00:00:00 from datamart/gold/feature_store/gold_feature_store_2024_05_01.parquet
X_inference (491, 57) | clickstream missing: 0.0
saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_05_01.parquet


---completed job---


2024-06-01


---starting job---


{'feature_store_directory': 'datamart/gold/feature_store/',
 'model_artefact_filepath': 'model_bank/cr

extracted features_sdf 530 2023-01-01 00:00:00 from datamart/gold/feature_store/gold_feature_store_2023_01_01.parquet


X_inference (530, 57) | clickstream missing: 0.0


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_01_01.parquet


---completed job---


2023-02-01


---starting job---


{'feature_store_directory': 'datamart/gold/feature_store/',
 'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 2, 1, 0, 0),
 'snapshot_date_str': '2023-02-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


... (51 outputs left out)

extracted features_sdf 515 2024-12-01 00:00:00 from datamart/gold/feature_store/gold_feature_store_2024_12_01.parquet
X_inference (515, 57) | clickstream missing: 1.0


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_12_01.parquet


---completed job---


2025-01-01


---starting job---


{'feature_store_directory': 'datamart/gold/feature_store/',
 'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2025, 1, 1, 0, 0),
 'snapshot_date_str': '2025-01-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


extracted features_sdf 526 2025-01-01 00:00:00 from datamart/gold/feature_store/gold_feature_store_2025_01_01.parquet
X_inference (526, 57) | clickstream missing: 1.0


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2025_01_01.parquet


---completed job---




## Check datamart

In [18]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [19]:
folder_path = "datamart/gold/model_predictions/" + model_name[:-4] + "/"
files_list = sorted(glob.glob(os.path.join(folder_path, '*.parquet')))
df = spark.read.parquet(*files_list)
print("row_count:", df.count())

df.show()

# one row per application month
df.groupBy("snapshot_date").agg(F.count("*").alias("n"), F.round(F.avg("model_predictions"), 3).alias("avg_score")) \
    .orderBy("snapshot_date").show(30)

row_count: 8974
+-----------+-------------+--------------------+--------------------+
|Customer_ID|snapshot_date|          model_name|   model_predictions|
+-----------+-------------+--------------------+--------------------+
| CUS_0x632f|   2023-02-01|credit_model_2024...|0.012357481755316257|
| CUS_0x6398|   2023-02-01|credit_model_2024...|   0.904067873954773|
| CUS_0x63ac|   2023-02-01|credit_model_2024...| 0.07488401234149933|
| CUS_0x63d4|   2023-02-01|credit_model_2024...| 0.20262040197849274|
| CUS_0x644f|   2023-02-01|credit_model_2024...| 0.06840576976537704|
| CUS_0x6473|   2023-02-01|credit_model_2024...| 0.18878740072250366|
| CUS_0x64aa|   2023-02-01|credit_model_2024...|  0.0177940521389246|
| CUS_0x64e5|   2023-02-01|credit_model_2024...| 0.05510004609823227|
| CUS_0x6563|   2023-02-01|credit_model_2024...|  0.4790586233139038|
| CUS_0x65aa|   2023-02-01|credit_model_2024...| 0.32292312383651733|
| CUS_0x6620|   2023-02-01|credit_model_2024...|  0.6694576740264893|
| CU

[Stage 119:===========================================>           (19 + 5) / 24]

+-------------+---+---------+
|snapshot_date|  n|avg_score|
+-------------+---+---------+
|   2023-01-01|530|    0.358|
|   2023-02-01|501|    0.389|
|   2023-03-01|506|    0.405|
|   2023-04-01|510|     0.37|
|   2023-05-01|521|     0.38|
|   2023-06-01|517|    0.347|
|   2023-07-01|471|    0.397|
|   2023-08-01|481|    0.382|
|   2023-09-01|454|    0.366|
|   2023-10-01|487|    0.365|
|   2023-11-01|491|    0.379|
|   2023-12-01|489|    0.392|
|   2024-01-01|485|    0.379|
|   2024-02-01|518|    0.404|
|   2024-03-01|511|    0.394|
|   2024-04-01|513|    0.384|
|   2024-05-01|491|    0.384|
|   2024-06-01|498|    0.379|
+-------------+---+---------+



+-----------+-------------+--------------------+-------------------+
|Customer_ID|snapshot_date|          model_name|  model_predictions|
+-----------+-------------+--------------------+-------------------+
| CUS_0x673b|   2024-08-01|credit_model_2024...|0.27979031205177307|
| CUS_0x67c8|   2024-08-01|credit_model_2024...|  0.302171915769577|
| CUS_0x682e|   2024-08-01|credit_model_2024...| 0.4223962128162384|
| CUS_0x6863|   2024-08-01|credit_model_2024...|  0.349252849817276|
| CUS_0x68cc|   2024-08-01|credit_model_2024...| 0.2241804450750351|
| CUS_0x68d5|   2024-08-01|credit_model_2024...| 0.2618422508239746|
| CUS_0x68da|   2024-08-01|credit_model_2024...| 0.5476460456848145|
| CUS_0x6938|   2024-08-01|credit_model_2024...|0.26298487186431885|
| CUS_0x693b|   2024-08-01|credit_model_2024...| 0.1768486201763153|
| CUS_0x6963|   2024-08-01|credit_model_2024...| 0.4140132963657379|
| CUS_0x6987|   2024-08-01|credit_model_2024...|0.18598143756389618|
| CUS_0x6991|   2024-08-01|credit_

+-------------+---+---------+
|snapshot_date|  n|avg_score|
+-------------+---+---------+
|   2023-01-01|530|    0.358|
|   2023-02-01|501|    0.389|
|   2023-03-01|506|    0.405|
|   2023-04-01|510|     0.37|
|   2023-05-01|521|     0.38|
|   2023-06-01|517|    0.347|
|   2023-07-01|471|    0.397|
|   2023-08-01|481|    0.382|
|   2023-09-01|454|    0.366|
|   2023-10-01|487|    0.365|
|   2023-11-01|491|    0.379|
|   2023-12-01|489|    0.392|
|   2024-01-01|485|    0.379|
|   2024-02-01|518|    0.404|
|   2024-03-01|511|    0.394|
|   2024-04-01|513|    0.384|
|   2024-05-01|491|    0.384|
|   2024-06-01|498|    0.379|
|   2024-07-01|505|     0.39|
|   2024-08-01|543|    0.391|
|   2024-09-01|493|    0.388|
|   2024-10-01|456|    0.377|
|   2024-11-01|488|    0.391|
|   2024-12-01|515|    0.365|
|   2025-01-01|526|    0.392|
+-------------+---+---------+

